# DataOrg Financial Health Prediction — Competition Solution
Maximises macro F1 on Low/Medium/High classification with XGBoost + LightGBM + CatBoost ensemble.

In [17]:
# ── Cell 1: Imports & Config ──────────────────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
from itertools import product as iproduct

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)

SEED   = 42
N_FOLD = 10
DATA_DIR = '.'

print('Libraries loaded OK')

Libraries loaded OK


In [18]:
# ── Cell 2: Load Data ─────────────────────────────────────────────────────────
train = pd.read_csv(os.path.join(DATA_DIR, 'Train.csv'))
test  = pd.read_csv(os.path.join(DATA_DIR, 'Test.csv'))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, 'SampleSubmission.csv'))

print(f'Train: {train.shape}  Test: {test.shape}')
print('Target distribution:')
print(train['Target'].value_counts())

Train: (9618, 39)  Test: (2405, 38)
Target distribution:
Target
Low       6280
Medium    2868
High       470
Name: count, dtype: int64


In [19]:
# ── Cell 3: Data Cleaning ─────────────────────────────────────────────────────

# Don't-know canonical token
DONTKNOW = "Dont_know"

# Patterns to unify as DONTKNOW (order matters — longer/more-specific first)
DONTKNOW_VARIANTS = [
    " Do not know / N\u200e/A",   # leading space + U+200E
    "Don\u2019t know or N/A",
    "Don't know or N/A",
    "Don\u2019t know (Do not show)",
    "Don't know (Do not show)",
    "Don?t know / doesn?t apply",
    "Don\u2019t Know",
    "Don't Know",
    "Don\u2019t know",
    "Don't know",
]


def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    obj_cols = df.select_dtypes('object').columns.tolist()

    # 3a. Normalise Unicode characters across all string columns
    for col in obj_cols:
        # curly apostrophe → straight
        df[col] = df[col].str.replace('\u2019', "'", regex=False)
        # invisible LTR mark
        df[col] = df[col].str.replace('\u200e', '', regex=False)
        # strip leading/trailing whitespace introduced by replacements
        df[col] = df[col].str.strip()

    # 3b. Harmonise all Don't-know variants → DONTKNOW
    # (re-build variants with straight apostrophe after unicode normalisation)
    dontknow_variants_normalised = [
        v.replace('\u2019', "'").replace('\u200e', '').strip()
        for v in DONTKNOW_VARIANTS
    ]
    dontknow_set = set(dontknow_variants_normalised)
    for col in obj_cols:
        df[col] = df[col].apply(
            lambda x: DONTKNOW if (isinstance(x, str) and x in dontknow_set) else x
        )

    # 3c. Column-specific fixes
    if 'current_problem_cash_flow' in df.columns:
        df['current_problem_cash_flow'] = df['current_problem_cash_flow'].replace({'0': 'No'})

    if 'compliance_income_tax' in df.columns:
        df['compliance_income_tax'] = df['compliance_income_tax'].replace({'Refused': DONTKNOW})

    # 3d. keeps_financial_records → ordinal numeric
    if 'keeps_financial_records' in df.columns:
        kfr_map = {
            'No': 0,
            'Yes': 1,
            'Yes, sometimes': 1,
            'Yes, always': 2,
        }
        df['keeps_financial_records'] = df['keeps_financial_records'].map(kfr_map)
        # NaN stays NaN (was already missing or unknown)

    return df


train = clean_df(train)
test  = clean_df(test)
print('Cleaning done.')
print('current_problem_cash_flow unique:', train['current_problem_cash_flow'].unique())
print('compliance_income_tax unique:', train['compliance_income_tax'].unique())
print('keeps_financial_records unique:', train['keeps_financial_records'].unique())

Cleaning done.
current_problem_cash_flow unique: <StringArray>
['Yes', nan, 'No']
Length: 3, dtype: str
compliance_income_tax unique: <StringArray>
['No', 'Yes', 'Dont_know', nan]
Length: 4, dtype: str
keeps_financial_records unique: [ 2.  0. nan  1.]


In [20]:
# ── Cell 4: Feature Engineering ───────────────────────────────────────────────

FIN_COLS   = ['personal_income', 'business_expenses', 'business_turnover']
HAVE_NOW_FIN = ['has_mobile_money', 'has_credit_card', 'has_loan_account',
                'has_internet_banking', 'has_debit_card']
HAVE_NOW_INS = ['motor_vehicle_insurance', 'medical_insurance', 'funeral_insurance']
POS_ATTITUDE = ['attitude_stable_business_environment',
                'attitude_satisfied_with_achievement',
                'attitude_more_successful_next_year',
                'perception_insurance_important']
NEG_ATTITUDE = ['attitude_worried_shutdown',
                'current_problem_cash_flow',
                'perception_cannot_afford_insurance']

# High-missing columns used for missingness signal
HIGH_MISSING_COLS = [
    'has_mobile_money', 'has_credit_card', 'has_loan_account',
    'has_internet_banking', 'has_debit_card', 'medical_insurance',
    'funeral_insurance', 'uses_friends_family_savings', 'uses_informal_lender',
    'motor_vehicle_insurance', 'future_risk_theft_stock', 'marketing_word_of_mouth',
    'problem_sourcing_money', 'covid_essential_service', 'motivation_make_more_money',
    'offers_credit_to_customers', 'current_problem_cash_flow',
]


def compute_country_stats(df_train: pd.DataFrame, cols):
    """Compute per-country median and std from training data."""
    stats = {}
    for col in cols:
        stats[col] = df_train.groupby('country')[col].agg(['median', 'std'])
    return stats


def engineer_features(df: pd.DataFrame, country_stats=None, is_train=True,
                      df_train_ref=None):
    df = df.copy()

    # ── Financial ratios ──────────────────────────────────────────────────────
    df['expense_to_turnover']  = df['business_expenses']  / (df['business_turnover'] + 1)
    df['profit_margin']        = (df['business_turnover'] - df['business_expenses']) / \
                                 (df['business_turnover'] + 1)
    df['income_to_expenses']   = df['personal_income']    / (df['business_expenses'] + 1)
    df['turnover_minus_expenses'] = df['business_turnover'] - df['business_expenses']

    df['log_personal_income']    = np.log1p(df['personal_income'])
    df['log_business_expenses']  = np.log1p(df['business_expenses'])
    df['log_business_turnover']  = np.log1p(df['business_turnover'])

    # ── Country-normalised financials ─────────────────────────────────────────
    if country_stats is None and is_train:
        country_stats = compute_country_stats(df, FIN_COLS)

    for col in FIN_COLS:
        stat = country_stats[col]
        medians = df['country'].map(stat['median'])
        stds    = df['country'].map(stat['std']).replace(0, 1).fillna(1)
        df[f'{col}_zscore'] = (df[col] - medians) / stds

        # Rank-percentile within country (for train: direct rank; for test: map via train quantiles)
        if is_train:
            df[f'{col}_pctrank'] = df.groupby('country')[col].rank(pct=True)
        else:
            # Use training CDF: for each test value, find fraction of train values <= it per country
            for country in df['country'].unique():
                mask_te = df['country'] == country
                if df_train_ref is not None:
                    mask_tr = df_train_ref['country'] == country
                    train_vals = df_train_ref.loc[mask_tr, col].dropna().values
                    if len(train_vals) > 0:
                        def _pct(v):
                            if pd.isna(v):
                                return np.nan
                            return float((train_vals <= v).mean())
                        df.loc[mask_te, f'{col}_pctrank'] = \
                            df.loc[mask_te, col].apply(_pct)
                    else:
                        df.loc[mask_te, f'{col}_pctrank'] = np.nan
                else:
                    df.loc[mask_te, f'{col}_pctrank'] = df.loc[mask_te, col].rank(pct=True)

    # ── Financial access score ────────────────────────────────────────────────
    for col in HAVE_NOW_FIN + HAVE_NOW_INS:
        if col in df.columns:
            flag_col = f'{col}_have_now'
            df[flag_col] = (df[col] == 'Have now').astype(float)
            df.loc[df[col].isna(), flag_col] = np.nan

    df['financial_product_count'] = df[[f'{c}_have_now' for c in HAVE_NOW_FIN
                                        if f'{c}_have_now' in df.columns]].sum(axis=1, min_count=1)
    df['insurance_product_count'] = df[[f'{c}_have_now' for c in HAVE_NOW_INS
                                        if f'{c}_have_now' in df.columns]].sum(axis=1, min_count=1)
    df['total_financial_access']  = df['financial_product_count'].fillna(0) + \
                                    df['insurance_product_count'].fillna(0)

    # ── Attitude/perception scores ────────────────────────────────────────────
    def yes_flag(series):
        return (series == 'Yes').astype(float).where(series.notna(), np.nan)

    pos_flags = [yes_flag(df[c]) for c in POS_ATTITUDE if c in df.columns]
    neg_flags = [yes_flag(df[c]) for c in NEG_ATTITUDE if c in df.columns]

    # current_problem_cash_flow was already cleaned to Yes/No or numeric — handle both
    # (It may have become numeric 0 after cleaning; re-check)
    if 'current_problem_cash_flow' in df.columns:
        cpf = df['current_problem_cash_flow']
        cpf_flag = (cpf == 'Yes').astype(float).where(cpf.notna(), np.nan)
        neg_flags = [yes_flag(df[c]) for c in NEG_ATTITUDE if c in df.columns
                     and c != 'current_problem_cash_flow']
        neg_flags.append(cpf_flag)

    if pos_flags:
        df['positive_attitude_count'] = pd.concat(pos_flags, axis=1).sum(axis=1, min_count=1)
    if neg_flags:
        df['negative_attitude_count'] = pd.concat(neg_flags, axis=1).sum(axis=1, min_count=1)
    if 'positive_attitude_count' in df.columns and 'negative_attitude_count' in df.columns:
        df['net_attitude'] = df['positive_attitude_count'].fillna(0) - \
                             df['negative_attitude_count'].fillna(0)

    # ── Missingness features ──────────────────────────────────────────────────
    existing_hm = [c for c in HIGH_MISSING_COLS if c in df.columns]
    df['missing_count']    = df[existing_hm].isna().sum(axis=1)
    df['missing_fraction'] = df.isna().sum(axis=1) / df.shape[1]

    # ── Business maturity ─────────────────────────────────────────────────────
    if 'business_age_years' in df.columns and 'business_age_months' in df.columns:
        df['total_business_months'] = df['business_age_years'] * 12 + \
                                      df['business_age_months'].fillna(0)
    if 'business_age_years' in df.columns:
        df['turnover_per_year']  = df['business_turnover'] / (df['business_age_years'] + 1)
        df['age_when_started']   = df['owner_age'] - df['business_age_years']

    # ── Interaction features ──────────────────────────────────────────────────
    if 'has_insurance' in df.columns:
        df['has_insurance_flag'] = (df['has_insurance'] == 'Yes').astype(float)
        df['has_insurance_x_log_turnover'] = df['has_insurance_flag'] * \
                                             df['log_business_turnover'].fillna(0)

    if 'keeps_financial_records' in df.columns:
        kfr = df['keeps_financial_records']
        df['keeps_records_always'] = (kfr == 2).astype(float).where(kfr.notna(), np.nan)
        df['keeps_records_always_x_log_turnover'] = \
            df['keeps_records_always'].fillna(0) * df['log_business_turnover'].fillna(0)

    if 'compliance_income_tax' in df.columns:
        df['compliance_yes'] = (df['compliance_income_tax'] == 'Yes').astype(float)
        if 'keeps_records_always' in df.columns:
            df['compliance_yes_x_keeps_records_always'] = \
                df['compliance_yes'].fillna(0) * df['keeps_records_always'].fillna(0)

    # Country × financial_product_count interaction (numeric)
    country_enc = {'eswatini': 1, 'lesotho': 2, 'malawi': 3, 'zimbabwe': 4}
    df['country_code'] = df['country'].map(country_enc).astype(float)
    df['country_x_financial_product_count'] = df['country_code'] * \
                                              df['financial_product_count'].fillna(0)

    return df, country_stats


train, country_stats = engineer_features(train, is_train=True)
test,  _             = engineer_features(test,  country_stats=country_stats,
                                         is_train=False, df_train_ref=train)

print(f'Train after FE: {train.shape}  Test after FE: {test.shape}')

Train after FE: (9618, 79)  Test after FE: (2405, 78)


In [26]:
# ── Cell 5: Encoding & Feature Split ─────────────────────────────────────────

TARGET_MAP = {'Low': 0, 'Medium': 1, 'High': 2}
INV_TARGET = {v: k for k, v in TARGET_MAP.items()}

y = train['Target'].map(TARGET_MAP).values

# Drop ID / Target from features
DROP_COLS = ['ID', 'Target']
feature_cols = [c for c in train.columns if c not in DROP_COLS]

# Identify original categorical columns (object dtype, excluding keeps_financial_records
# which is now numeric)
cat_cols_original = [c for c in feature_cols
                     if not pd.api.types.is_numeric_dtype(train[c])]
num_cols = [c for c in feature_cols if c not in cat_cols_original]

print(f'Categorical cols: {len(cat_cols_original)}')
print(f'Numeric cols: {len(num_cols)}')
print(f'Total features: {len(feature_cols)}')

# ── Build CatBoost matrices (raw strings, NaN → 'MISSING') ──────────────────
def prep_cb(df, cols, cat_cols):
    X = df[cols].copy()
    for c in cat_cols:
        X[c] = X[c].fillna('MISSING').astype(str)
    return X

X_cb_full_train = prep_cb(train, feature_cols, cat_cols_original)
X_cb_test       = prep_cb(test,  feature_cols, cat_cols_original)
cat_feature_indices = [X_cb_full_train.columns.get_loc(c) for c in cat_cols_original]

# ── Build XGBoost / LightGBM matrices (label-encoded, NaN preserved) ────────
combined = pd.concat([train[feature_cols], test[feature_cols]], axis=0, ignore_index=True)

label_encoders = {}
for col in cat_cols_original:
    le = LabelEncoder()
    combined[col] = combined[col].astype(str)  # NaN → 'nan' string
    le.fit(combined[col])
    combined[col] = le.transform(combined[col]).astype(float)
    # Re-assign nan label back to NaN
    if 'nan' in le.classes_:
        nan_label = le.transform(['nan'])[0]
        combined.loc[combined[col] == nan_label, col] = np.nan
    label_encoders[col] = le

n_train = len(train)
X_le_train = combined.iloc[:n_train].values.astype(float)
X_le_test  = combined.iloc[n_train:].values.astype(float)

print('Encoding done.')
print(f'X_le_train: {X_le_train.shape}  X_le_test: {X_le_test.shape}')

Categorical cols: 30
Numeric cols: 47
Total features: 77
Encoding done.
X_le_train: (9618, 77)  X_le_test: (2405, 77)


In [ ]:
# ── Cell 6: XGBoost 10-Fold CV ───────────────────────────────────────────────

classes = np.array([0, 1, 2])
class_weights_arr = compute_class_weight('balanced', classes=classes, y=y)
class_weight_dict = {0: class_weights_arr[0],
                     1: class_weights_arr[1],
                     2: class_weights_arr[2]}

params_xgb = dict(
    max_depth=7,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_weight=5,
    gamma=0.1,
    reg_alpha=0.5,
    reg_lambda=1.0,
    n_estimators=2000,
    tree_method='hist',
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=SEED,
    verbosity=0,
)

skf = StratifiedKFold(n_splits=N_FOLD, shuffle=True, random_state=SEED)

oof_xgb  = np.zeros((n_train, 3))
test_xgb = np.zeros((len(test), 3))

print('Training XGBoost...')
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_le_train, y)):
    X_tr, X_va = X_le_train[tr_idx], X_le_train[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    sw = np.array([class_weight_dict[yi] for yi in y_tr])

    dtrain = xgb.DMatrix(X_tr, label=y_tr, weight=sw)
    dval = xgb.DMatrix(X_va, label=y_va)
    dtest = xgb.DMatrix(X_le_test)

    model = xgb.train(
        params_xgb,
        dtrain,
        num_boost_round=2000,
        evals=[(dval, 'val')],
        early_stopping_rounds=100,
        verbose_eval=False,
    )
    oof_xgb[va_idx]  = model.predict(dval)
    test_xgb        += model.predict(dtest) / N_FOLD

    fold_f1 = f1_score(y_va, oof_xgb[va_idx].argmax(1), average='macro')
    print(f'  Fold {fold+1:2d} | best_iter={model.best_iteration:4d} | F1={fold_f1:.4f}')

xgb_oof_f1 = f1_score(y, oof_xgb.argmax(1), average='macro')
print(f'\nXGBoost OOF Macro F1: {xgb_oof_f1:.4f}')

Training XGBoost...


ValueError: too many values to unpack (expected 2)

In [ ]:
# ── Cell 7: LightGBM 10-Fold CV ──────────────────────────────────────────────

params_lgb = dict(
    max_depth=8,
    learning_rate=0.03,
    num_leaves=63,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.7,
    min_child_samples=20,
    reg_alpha=0.3,
    reg_lambda=0.5,
    n_estimators=2000,
    is_unbalance=True,
    objective='multiclass',
    num_class=3,
    metric='multi_logloss',
    random_state=SEED,
    verbose=-1,
    n_jobs=-1,
)

oof_lgb  = np.zeros((n_train, 3))
test_lgb = np.zeros((len(test), 3))
lgb_models = []

print('Training LightGBM...')
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_le_train, y)):
    X_tr, X_va = X_le_train[tr_idx], X_le_train[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    model = lgb.LGBMClassifier(**params_lgb)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(100, verbose=False),
                   lgb.log_evaluation(period=-1)],
    )
    oof_lgb[va_idx]  = model.predict_proba(X_va)
    test_lgb        += model.predict_proba(X_le_test) / N_FOLD
    lgb_models.append(model)

    fold_f1 = f1_score(y_va, oof_lgb[va_idx].argmax(1), average='macro')
    print(f'  Fold {fold+1:2d} | best_iter={model.best_iteration_:4d} | F1={fold_f1:.4f}')

lgb_oof_f1 = f1_score(y, oof_lgb.argmax(1), average='macro')
print(f'\nLightGBM OOF Macro F1: {lgb_oof_f1:.4f}')

In [ ]:
# ── Cell 8: CatBoost 10-Fold CV ───────────────────────────────────────────────

params_cb = dict(
    depth=8,
    learning_rate=0.03,
    l2_leaf_reg=3,
    iterations=2000,
    early_stopping_rounds=100,
    loss_function='MultiClass',
    eval_metric='TotalF1:average=Macro',
    auto_class_weights='Balanced',
    random_seed=SEED,
    verbose=0,
    thread_count=-1,
)

oof_cb  = np.zeros((n_train, 3))
test_cb = np.zeros((len(test), 3))

X_cb_full_arr  = X_cb_full_train.values
X_cb_test_arr  = X_cb_test.values

print('Training CatBoost...')
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_cb_full_arr, y)):
    X_tr, X_va = X_cb_full_arr[tr_idx], X_cb_full_arr[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    train_pool = Pool(X_tr, label=y_tr, cat_features=cat_feature_indices)
    val_pool   = Pool(X_va, label=y_va, cat_features=cat_feature_indices)
    test_pool  = Pool(X_cb_test_arr,    cat_features=cat_feature_indices)

    model = CatBoostClassifier(**params_cb)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)

    oof_cb[va_idx]  = model.predict_proba(val_pool)
    test_cb        += model.predict_proba(test_pool) / N_FOLD

    fold_f1 = f1_score(y_va, oof_cb[va_idx].argmax(1), average='macro')
    print(f'  Fold {fold+1:2d} | best_iter={model.best_iteration_:4d} | F1={fold_f1:.4f}')

cb_oof_f1 = f1_score(y, oof_cb.argmax(1), average='macro')
print(f'\nCatBoost OOF Macro F1: {cb_oof_f1:.4f}')

In [ ]:
# ── Cell 9: Ensemble Weight Optimisation ──────────────────────────────────────

best_w  = (1/3, 1/3, 1/3)
best_f1 = -1

step = 0.05
grid = np.arange(0, 1 + step, step)

print('Searching ensemble weights...')
for w1 in grid:
    for w2 in grid:
        w3 = round(1 - w1 - w2, 10)
        if w3 < -1e-9 or w3 > 1 + 1e-9:
            continue
        w3 = max(0, min(1, w3))
        blend = w1 * oof_xgb + w2 * oof_lgb + w3 * oof_cb
        score = f1_score(y, blend.argmax(1), average='macro')
        if score > best_f1:
            best_f1 = score
            best_w  = (w1, w2, w3)

w1, w2, w3 = best_w
print(f'\nBest ensemble weights: XGB={w1:.2f}  LGB={w2:.2f}  CB={w3:.2f}')
print(f'Blended OOF Macro F1 (untuned): {best_f1:.4f}')
print(f'  XGBoost OOF F1 : {xgb_oof_f1:.4f}')
print(f'  LightGBM OOF F1: {lgb_oof_f1:.4f}')
print(f'  CatBoost OOF F1: {cb_oof_f1:.4f}')

blend_oof  = w1 * oof_xgb  + w2 * oof_lgb  + w3 * oof_cb
blend_test = w1 * test_xgb + w2 * test_lgb + w3 * test_cb

In [ ]:
# ── Cell 10: Post-Prediction Threshold Tuning ─────────────────────────────────
# Scale class probabilities to shift decision boundaries toward minority classes

best_high_boost = 1.0
best_med_boost  = 1.0
best_tune_f1    = best_f1

print('Searching threshold boosts...')
for high_boost in np.arange(1.0, 3.1, 0.1):
    for med_boost in np.arange(0.8, 1.55, 0.05):
        scaled = blend_oof.copy()
        scaled[:, 1] *= med_boost
        scaled[:, 2] *= high_boost
        preds = scaled.argmax(1)
        score = f1_score(y, preds, average='macro')
        if score > best_tune_f1:
            best_tune_f1    = score
            best_high_boost = high_boost
            best_med_boost  = med_boost

print(f'Best high_boost={best_high_boost:.1f}  med_boost={best_med_boost:.2f}')
print(f'OOF Macro F1 after threshold tuning: {best_tune_f1:.4f}')

# Apply to OOF for reporting
scaled_oof = blend_oof.copy()
scaled_oof[:, 1] *= best_med_boost
scaled_oof[:, 2] *= best_high_boost
final_oof_preds = scaled_oof.argmax(1)

In [ ]:
# ── Cell 11: Final Test Predictions & Submission ──────────────────────────────

scaled_test = blend_test.copy()
scaled_test[:, 1] *= best_med_boost
scaled_test[:, 2] *= best_high_boost
final_test_preds = scaled_test.argmax(1)

pred_labels = [INV_TARGET[p] for p in final_test_preds]

submission = pd.DataFrame({'ID': test['ID'], 'Target': pred_labels})
submission.to_csv(os.path.join(DATA_DIR, 'submission.csv'), index=False)
print('submission.csv saved!')
print(f'Shape: {submission.shape}')

In [ ]:
# ── Cell 12: Diagnostics & Feature Importance ─────────────────────────────────

print('=' * 60)
print('MODEL PERFORMANCE SUMMARY')
print('=' * 60)
print(f'XGBoost  OOF Macro F1 : {xgb_oof_f1:.4f}')
print(f'LightGBM OOF Macro F1 : {lgb_oof_f1:.4f}')
print(f'CatBoost OOF Macro F1 : {cb_oof_f1:.4f}')
print(f'Ensemble (untuned)    : {best_f1:.4f}')
print(f'Ensemble (tuned)      : {best_tune_f1:.4f}')
print(f'Ensemble weights      : XGB={w1:.2f} LGB={w2:.2f} CB={w3:.2f}')
print(f'High boost            : {best_high_boost:.1f}')
print(f'Med boost             : {best_med_boost:.2f}')

print('\n' + '=' * 60)
print('OOF CLASSIFICATION REPORT (tuned ensemble)')
print('=' * 60)
print(classification_report(
    y, final_oof_preds,
    target_names=['Low', 'Medium', 'High']
))

print('=' * 60)
print('SUBMISSION TARGET DISTRIBUTION')
print('=' * 60)
dist = submission['Target'].value_counts()
print(dist)
print(f'Total rows: {len(submission)}')

print('\n' + '=' * 60)
print('TOP 30 FEATURES BY LIGHTGBM GAIN')
print('=' * 60)
# Average importances across folds
importance_df = pd.DataFrame(index=feature_cols)
for i, m in enumerate(lgb_models):
    importance_df[f'fold_{i}'] = m.booster_.feature_importance(importance_type='gain')
importance_df['mean_gain'] = importance_df.mean(axis=1)
top30 = importance_df['mean_gain'].sort_values(ascending=False).head(30)
for feat, val in top30.items():
    print(f'  {feat:<50s} {val:>12.1f}')